In [15]:
# ============================================================
# PFE Pruning Experiments v8 — Corrected & Production-Ready
#
# Changes from v7:
#   - Method 8: proper online movement pruning (Sanh et al. 2020)
#     with per-parameter score tensors updated each backward pass
#   - Multi-sparsity sweep for all mask-based methods (unstructured,
#     magnitude, iterative, one-shot, movement) and multi-ratio sweep
#     for structured pruning; only the best sparsity/ratio per method
#     is saved to JSON
#   - Timing follows the ViT baseline exactly:
#       CPU  — single image (100 reps) + batch-128 (20 reps)
#       GPU  — single image synchronized (500 reps) +
#              batch-128 CUDA events (100 reps)
#   - Full config block printed at startup and saved inside JSON
# ============================================================

import torch, torchvision, time, os, json, copy, random
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
from thop import profile
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)

In [16]:
# ── REPRODUCIBILITY ───────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

def seed_worker(wid):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

g = torch.Generator(); g.manual_seed(SEED)

In [17]:
# ── CONFIG ────────────────────────────────────────────────────
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
BASELINE   = "../__2__baseline_resnet50_cifar10.pth"
OUTPUT_JSON = "__3__pruning_results_v8.json"
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# Sparsity levels swept for all mask-based methods
SPARSITY_LEVELS = [0.30, 0.50, 0.70, 0.80, 0.90]

# Channel prune ratios swept for structured pruning
CHANNEL_RATIOS = [0.20, 0.30, 0.40, 0.50]

BASELINE_EPOCHS = 50
FINETUNE_EPOCHS = 10
FINETUNE_LR     = 1e-3

N_ROUNDS       = 3
_base_ft       = FINETUNE_EPOCHS // N_ROUNDS
ITER_FT_EPOCHS = [_base_ft + (1 if r == 0 else 0) for r in range(N_ROUNDS)]   # [4,3,3]
assert sum(ITER_FT_EPOCHS) == FINETUNE_EPOCHS

LTH_TRAIN_EPOCHS   = BASELINE_EPOCHS
LTH_RETRAIN_EPOCHS = BASELINE_EPOCHS

# Timing config — mirrors ViT baseline script exactly
CPU_WARMUP_SINGLE  = 10
CPU_REPS_SINGLE    = 100
CPU_WARMUP_BATCH   = 5
CPU_REPS_BATCH     = 20
GPU_WARMUP_SINGLE  = 50
GPU_REPS_SINGLE    = 500
GPU_WARMUP_BATCH   = 10
GPU_REPS_BATCH     = 100

CONFIG = dict(
    device              = str(DEVICE),
    seed                = SEED,
    batch_size          = BATCH_SIZE,
    baseline_pth        = BASELINE,
    sparsity_levels     = SPARSITY_LEVELS,
    channel_ratios      = CHANNEL_RATIOS,
    baseline_epochs     = BASELINE_EPOCHS,
    finetune_epochs     = FINETUNE_EPOCHS,
    finetune_lr         = FINETUNE_LR,
    iterative_rounds    = N_ROUNDS,
    iterative_ft_epochs = ITER_FT_EPOCHS,
    lth_train_epochs    = LTH_TRAIN_EPOCHS,
    lth_retrain_epochs  = LTH_RETRAIN_EPOCHS,
    timing = dict(
        cpu_warmup_single = CPU_WARMUP_SINGLE,
        cpu_reps_single   = CPU_REPS_SINGLE,
        cpu_warmup_batch  = CPU_WARMUP_BATCH,
        cpu_reps_batch    = CPU_REPS_BATCH,
        gpu_warmup_single = GPU_WARMUP_SINGLE,
        gpu_reps_single   = GPU_REPS_SINGLE,
        gpu_warmup_batch  = GPU_WARMUP_BATCH,
        gpu_reps_batch    = GPU_REPS_BATCH,
        method            = "CPU: time.perf_counter() | GPU single: cuda.synchronize() | GPU batch: CUDA events",
    ),
)

print("\n" + "="*65)
print("CONFIGURATION")
print("="*65)
for k, v in CONFIG.items():
    if k == "timing":
        print(f"  timing:")
        for tk, tv in v.items():
            print(f"    {tk:<22}: {tv}")
    else:
        print(f"  {k:<22}: {v}")
print()


CONFIGURATION
  device                : cuda
  seed                  : 42
  batch_size            : 128
  baseline_pth          : ../__2__baseline_resnet50_cifar10.pth
  sparsity_levels       : [0.3, 0.5, 0.7, 0.8, 0.9]
  channel_ratios        : [0.2, 0.3, 0.4, 0.5]
  baseline_epochs       : 50
  finetune_epochs       : 10
  finetune_lr           : 0.001
  iterative_rounds      : 3
  iterative_ft_epochs   : [4, 3, 3]
  lth_train_epochs      : 50
  lth_retrain_epochs    : 50
  timing:
    cpu_warmup_single     : 10
    cpu_reps_single       : 100
    cpu_warmup_batch      : 5
    cpu_reps_batch        : 20
    gpu_warmup_single     : 50
    gpu_reps_single       : 500
    gpu_warmup_batch      : 10
    gpu_reps_batch        : 100
    method                : CPU: time.perf_counter() | GPU single: cuda.synchronize() | GPU batch: CUDA events



In [18]:
# ── MODEL FACTORY ─────────────────────────────────────────────
def build_model(num_classes=NUM_CLASSES):
    m = models.resnet50(pretrained=False)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, num_classes)
    return m

def load_baseline():
    m = build_model()
    m.load_state_dict(torch.load(BASELINE, map_location=DEVICE))
    return m.to(DEVICE)

In [19]:
# ── DATA ──────────────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_set    = torchvision.datasets.CIFAR10('../data', True,  download=True,
                                            transform=transform_train)
test_set     = torchvision.datasets.CIFAR10('../data', False, download=True,
                                            transform=transform_test)
train_loader = torch.utils.data.DataLoader(
    train_set, BATCH_SIZE, shuffle=True,  num_workers=0,
    worker_init_fn=seed_worker, generator=g)
test_loader  = torch.utils.data.DataLoader(
    test_set,  BATCH_SIZE, shuffle=False, num_workers=0)

e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [20]:
# ── METRIC HELPERS ────────────────────────────────────────────
def evaluate_full(model):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            preds.extend(model(x.to(DEVICE)).argmax(1).cpu().numpy())
            labels.extend(y.numpy())
    p, l = np.array(preds), np.array(labels)
    return dict(
        accuracy  = float(accuracy_score(l, p)),
        precision = float(precision_score(l, p, average='macro', zero_division=0)),
        recall    = float(recall_score(l, p, average='macro', zero_division=0)),
        f1        = float(f1_score(l, p, average='macro', zero_division=0)),
    )

def count_params(model):  return int(sum(p.numel() for p in model.parameters()))
def count_nonzero(model): return int(sum((p != 0).sum().item() for p in model.parameters()))

def model_size_mb(model, path="_tmp_.pth"):
    torch.save(model.state_dict(), path)
    mb = os.path.getsize(path) / 1e6
    os.remove(path)
    return float(mb)

def theoretical_size_mb(model):
    nz = count_nonzero(model)
    return round((nz * 4) / 1e6, 4)

def compute_flops(model):
    model.eval()
    dummy = torch.randn(1, 3, 32, 32).to(DEVICE)
    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    return float(macs * 2 / 1e9)

In [21]:
# ── TIMING — mirrors ViT baseline script exactly ──────────────
def measure_inference_times(model):
    """
    Returns a dict with CPU and GPU timing, matching the ViT baseline
    script structure exactly. GPU timing uses CUDA events for batch
    and synchronized perf_counter for single-image.
    """
    timing = {}
    is_gpu = DEVICE.type == "cuda"

    # ── CPU timing ────────────────────────────────────────────
    print("    ⏱  CPU timing ...")
    model_cpu = copy.deepcopy(model).cpu().eval()
    dummy_single_cpu = torch.randn(1, 3, 32, 32)
    dummy_batch_cpu  = torch.randn(BATCH_SIZE, 3, 32, 32)

    with torch.no_grad():
        for _ in range(CPU_WARMUP_SINGLE):
            model_cpu(dummy_single_cpu)
    times_cpu_single = []
    with torch.no_grad():
        for _ in range(CPU_REPS_SINGLE):
            t0 = time.perf_counter()
            model_cpu(dummy_single_cpu)
            times_cpu_single.append(time.perf_counter() - t0)
    inf_ms_single_cpu = float(np.mean(times_cpu_single) * 1000)

    with torch.no_grad():
        for _ in range(CPU_WARMUP_BATCH):
            model_cpu(dummy_batch_cpu)
    times_cpu_batch = []
    with torch.no_grad():
        for _ in range(CPU_REPS_BATCH):
            t0 = time.perf_counter()
            model_cpu(dummy_batch_cpu)
            times_cpu_batch.append(time.perf_counter() - t0)
    inf_ms_batch_cpu = float(np.mean(times_cpu_batch) * 1000)

    timing["cpu"] = dict(
        single_img_ms       = round(inf_ms_single_cpu, 4),
        batch128_ms         = round(inf_ms_batch_cpu, 4),
        per_img_ms          = round(inf_ms_batch_cpu / BATCH_SIZE, 4),
        throughput_imgs_sec = round(BATCH_SIZE / (inf_ms_batch_cpu / 1000), 1),
        timing_method       = "time.perf_counter()",
    )
    del model_cpu, dummy_single_cpu, dummy_batch_cpu

    # ── GPU timing ────────────────────────────────────────────
    if is_gpu:
        print("    ⏱  GPU timing ...")
        model.eval()
        dummy_single_gpu = torch.randn(1, 3, 32, 32, device=DEVICE)
        dummy_batch_gpu  = torch.randn(BATCH_SIZE, 3, 32, 32, device=DEVICE)

        # single image — synchronized perf_counter
        with torch.no_grad():
            for _ in range(GPU_WARMUP_SINGLE):
                model(dummy_single_gpu)
        torch.cuda.synchronize()
        times_gpu_single = []
        with torch.no_grad():
            for _ in range(GPU_REPS_SINGLE):
                torch.cuda.synchronize()
                t0 = time.perf_counter()
                model(dummy_single_gpu)
                torch.cuda.synchronize()
                times_gpu_single.append(time.perf_counter() - t0)
        inf_ms_single_gpu = float(np.mean(times_gpu_single) * 1000)

        # batch — CUDA events
        start_ev = torch.cuda.Event(enable_timing=True)
        end_ev   = torch.cuda.Event(enable_timing=True)
        with torch.no_grad():
            for _ in range(GPU_WARMUP_BATCH):
                model(dummy_batch_gpu)
        torch.cuda.synchronize()
        times_gpu_batch = []
        with torch.no_grad():
            for _ in range(GPU_REPS_BATCH):
                start_ev.record()
                model(dummy_batch_gpu)
                end_ev.record()
                torch.cuda.synchronize()
                times_gpu_batch.append(start_ev.elapsed_time(end_ev))
        inf_ms_batch_gpu = float(np.mean(times_gpu_batch))

        timing["gpu"] = dict(
            single_img_ms       = round(inf_ms_single_gpu, 4),
            batch128_ms         = round(inf_ms_batch_gpu, 4),
            per_img_ms          = round(inf_ms_batch_gpu / BATCH_SIZE, 4),
            throughput_imgs_sec = round(BATCH_SIZE / (inf_ms_batch_gpu / 1000), 1),
            timing_method       = "single: cuda.synchronize()+perf_counter | batch: CUDA events",
        )
        del dummy_single_gpu, dummy_batch_gpu
    else:
        timing["gpu"] = None

    return timing

def collect_metrics(model, sparsity_target=None, flops_note=None):
    m   = evaluate_full(model)
    tot = count_params(model)
    nz  = count_nonzero(model)
    m.update(dict(
        params_total        = tot,
        params_nonzero      = nz,
        sparsity_actual     = float(1 - nz / tot),
        sparsity_target     = sparsity_target,
        size_mb             = model_size_mb(model),
        size_mb_theoretical = theoretical_size_mb(model),
        flops_G             = compute_flops(model),
        inference_ms        = None,   # filled in timing pass
    ))
    if flops_note:
        m["flops_note"] = flops_note
    return m

FLOPS_NOTE_MASK = (
    "Mask-based pruning — computation graph unchanged. "
    "Dense FLOPs identical to baseline. "
    "Effective FLOPs require sparse-kernel backend (e.g. DeepSparse)."
)

In [22]:
# ── TRAINING UTILITIES ────────────────────────────────────────
def train_model(model, epochs, lr=0.1, desc="train", frozen_mask=None):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr,
                                momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()
            if frozen_mask is not None:
                with torch.no_grad():
                    for name, p in model.named_parameters():
                        if name in frozen_mask:
                            p.mul_(frozen_mask[name])
        scheduler.step()
        if (ep + 1) % 10 == 0 or ep == 0:
            acc = evaluate_full(model)["accuracy"]
            print(f"    [{desc}] ep {ep+1:>3}/{epochs}  acc={acc:.4f}")
    return model

def fine_tune(model, epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR,
              desc="ft", frozen_mask=None):
    return train_model(model, epochs, lr=lr, desc=desc, frozen_mask=frozen_mask)

def get_prunable_layers(model):
    return [(mod, 'weight') for mod in model.modules()
            if isinstance(mod, (nn.Conv2d, nn.Linear))]

def make_permanent(model):
    for mod, _ in get_prunable_layers(model):
        try:  prune.remove(mod, 'weight')
        except ValueError: pass
    return model

def get_conv_linear_weight_keys(model):
    keys = []
    for name, mod in model.named_modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)):
            key = f"{name}.weight" if name else "weight"
            keys.append(key)
    assert len(keys) >= 50, f"Expected ≥50 keys, got {len(keys)}"
    return keys

In [23]:
# ── CHECKPOINT SAVING ─────────────────────────────────────────
SAVE_DIR = "saved_models_v8"
os.makedirs(SAVE_DIR, exist_ok=True)

def save_model(model, key):
    path = os.path.join(SAVE_DIR, f"{key}.pth")
    torch.save(model.state_dict(), path)
    print(f"  ✓ Saved → {path}")

In [24]:
# ── STRUCTURED PRUNING HELPERS ────────────────────────────────
def _prune_conv_out(conv, kept_idx):
    n   = len(kept_idx)
    new = nn.Conv2d(conv.in_channels, n, conv.kernel_size, conv.stride,
                    conv.padding, groups=conv.groups,
                    bias=(conv.bias is not None))
    new.weight.data = conv.weight.data[kept_idx].clone()
    if conv.bias is not None:
        new.bias.data = conv.bias.data[kept_idx].clone()
    return new

def _prune_conv_in(conv, kept_idx):
    n   = len(kept_idx)
    new = nn.Conv2d(n, conv.out_channels, conv.kernel_size, conv.stride,
                    conv.padding, groups=conv.groups,
                    bias=(conv.bias is not None))
    new.weight.data = conv.weight.data[:, kept_idx].clone()
    if conv.bias is not None:
        new.bias.data = conv.bias.data.clone()
    return new

def _prune_bn(bn, kept_idx):
    n   = len(kept_idx)
    new = nn.BatchNorm2d(n, eps=bn.eps, momentum=bn.momentum,
                         affine=bn.affine,
                         track_running_stats=bn.track_running_stats)
    if bn.affine:
        new.weight.data = bn.weight.data[kept_idx].clone()
        new.bias.data   = bn.bias.data[kept_idx].clone()
    if bn.track_running_stats:
        new.running_mean        = bn.running_mean[kept_idx].clone()
        new.running_var         = bn.running_var[kept_idx].clone()
        new.num_batches_tracked = bn.num_batches_tracked.clone()
    return new

def prune_resnet50_structured(model, ratio):
    model = copy.deepcopy(model)
    for stage in ['layer1', 'layer2', 'layer3', 'layer4']:
        for block in getattr(model, stage):
            conv1  = block.conv1
            bn1    = block.bn1
            conv2  = block.conv2
            scores = conv1.weight.data.view(conv1.out_channels, -1).norm(p=2, dim=1)
            n_keep = max(1, int(conv1.out_channels * (1 - ratio)))
            kept   = scores.argsort(descending=True)[:n_keep].sort().values
            block.conv1 = _prune_conv_out(conv1, kept)
            block.bn1   = _prune_bn(bn1, kept)
            block.conv2 = _prune_conv_in(conv2, kept)
            assert block.conv1.out_channels == block.conv2.in_channels
    return model

In [25]:
# ── STORAGE: final results and model registry ─────────────────
results        = {}
model_registry = {}

In [26]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SWEEP UTILITY                                               ║
# ║  Runs a method across all sparsity levels and keeps the best ║
# ╚══════════════════════════════════════════════════════════════╝
def run_sparsity_sweep(method_fn, sparsity_levels, result_key,
                       method_note_fn=None, flops_note=FLOPS_NOTE_MASK):
    """
    Calls method_fn(sparsity) -> (model, metrics_dict) for every level.
    Keeps the result with the highest accuracy.
    Returns (best_model, best_metrics, best_sparsity).
    """
    best_acc      = -1.0
    best_model    = None
    best_metrics  = None
    best_sparsity = None
    sweep_log     = []

    for sp in sparsity_levels:
        print(f"\n  → sparsity={sp:.0%} ...")
        m_sp, metrics_sp = method_fn(sp)
        acc_sp = metrics_sp["accuracy"]
        sweep_log.append({"sparsity": sp, "accuracy": acc_sp,
                           "f1": metrics_sp["f1"]})
        print(f"     acc={acc_sp:.4f}  sparsity_actual={metrics_sp['sparsity_actual']:.3f}")
        if acc_sp > best_acc:
            best_acc      = acc_sp
            best_model    = m_sp
            best_metrics  = metrics_sp
            best_sparsity = sp

    print(f"\n  ★ Best sparsity for {result_key}: {best_sparsity:.0%}  "
          f"(acc={best_acc:.4f})")
    best_metrics["best_sparsity_target"] = best_sparsity
    best_metrics["sparsity_sweep_log"]   = sweep_log
    if flops_note:
        best_metrics["flops_note"] = flops_note
    if method_note_fn:
        best_metrics["method_note"] = method_note_fn(best_sparsity)
    return best_model, best_metrics, best_sparsity

In [27]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  METHOD 1a — UNSTRUCTURED  (global L1, no fine-tune)         ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "="*65)
print("1a. UNSTRUCTURED — global L1, no fine-tune  (sweep)")
print("="*65)

def run_1a(sparsity):
    m = load_baseline()
    prune.global_unstructured(get_prunable_layers(m),
                               pruning_method=prune.L1Unstructured,
                               amount=sparsity)
    make_permanent(m)
    return m, collect_metrics(m, sparsity_target=sparsity)

model_1a, results["1a_unstructured_noft"], best_sp_1a = run_sparsity_sweep(
    run_1a, SPARSITY_LEVELS, "1a_unstructured_noft",
    method_note_fn=lambda sp: (
        f"Global L1 unstructured @ {sp:.0%} sparsity, no fine-tuning. "
        "Best sparsity selected by accuracy across sweep. "
        "Paired with 7_one_shot (per-layer, no ft) → isolates threshold scope. "
        "Paired with 1b_unstructured_ft (same pruning, adds ft) → isolates ft value."
    ),
)
model_registry["1a_unstructured_noft"] = model_1a
save_model(model_1a, "1a_unstructured_noft")


1a. UNSTRUCTURED — global L1, no fine-tune  (sweep)

  → sparsity=30% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


     acc=0.9321  sparsity_actual=0.299

  → sparsity=50% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


     acc=0.9320  sparsity_actual=0.499

  → sparsity=70% ...


...

     acc=0.9319  sparsity_actual=0.698

  → sparsity=80% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


     acc=0.9276  sparsity_actual=0.798

  → sparsity=90% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


     acc=0.8774  sparsity_actual=0.898

  ★ Best sparsity for 1a_unstructured_noft: 30%  (acc=0.9321)
  ✓ Saved → saved_models_v8\1a_unstructured_noft.pth


In [28]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  METHOD 1b — UNSTRUCTURED  (global L1, with fine-tune)       ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "="*65)
print("1b. UNSTRUCTURED — global L1, with fine-tune  (sweep)")
print("="*65)

def run_1b(sparsity):
    m = load_baseline()
    prune.global_unstructured(get_prunable_layers(m),
                               pruning_method=prune.L1Unstructured,
                               amount=sparsity)
    m = fine_tune(m, epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR, desc="1b-ft")
    make_permanent(m)
    return m, collect_metrics(m, sparsity_target=sparsity)

model_1b, results["1b_unstructured_ft"], best_sp_1b = run_sparsity_sweep(
    run_1b, SPARSITY_LEVELS, "1b_unstructured_ft",
    method_note_fn=lambda sp: (
        f"Global L1 unstructured @ {sp:.0%} sparsity + {FINETUNE_EPOCHS}-epoch "
        f"fine-tune @ lr={FINETUNE_LR}. Best sparsity selected by accuracy. "
        "Paired with 1a (isolates ft value) and 3_magnitude (isolates threshold scope)."
    ),
)
model_registry["1b_unstructured_ft"] = model_1b
save_model(model_1b, "1b_unstructured_ft")


1b. UNSTRUCTURED — global L1, with fine-tune  (sweep)

  → sparsity=30% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [1b-ft] ep   1/10  acc=0.9310
    [1b-ft] ep  10/10  acc=0.9314
     acc=0.9314  sparsity_actual=0.299

  → sparsity=50% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [1b-ft] ep   1/10  acc=0.9303
    [1b-ft] ep  10/10  acc=0.9325
     acc=0.9325  sparsity_actual=0.499

  → sparsity=70% ...


...

    [1b-ft] ep   1/10  acc=0.9304
    [1b-ft] ep  10/10  acc=0.9313
     acc=0.9313  sparsity_actual=0.698

  → sparsity=80% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [1b-ft] ep   1/10  acc=0.9304
    [1b-ft] ep  10/10  acc=0.9303
     acc=0.9303  sparsity_actual=0.798

  → sparsity=90% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [1b-ft] ep   1/10  acc=0.9291
    [1b-ft] ep  10/10  acc=0.9314
     acc=0.9314  sparsity_actual=0.898

  ★ Best sparsity for 1b_unstructured_ft: 50%  (acc=0.9325)
  ✓ Saved → saved_models_v8\1b_unstructured_ft.pth


In [29]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  METHOD 2 — STRUCTURED  (channel removal, Bottleneck-safe)   ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "="*65)
print("2. STRUCTURED — channel pruning  (ratio sweep)")
print("="*65)

best_acc_2    = -1.0
best_model_2  = None
best_metrics_2 = None
best_ratio_2  = None
sweep_log_2   = []

for ratio in CHANNEL_RATIOS:
    print(f"\n  → channel_ratio={ratio:.0%} ...")
    m_r = prune_resnet50_structured(load_baseline(), ratio=ratio).to(DEVICE)
    m_r = fine_tune(m_r, epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR,
                    desc=f"2-structured-{ratio:.0%}")
    metrics_r = collect_metrics(m_r, sparsity_target=ratio)
    metrics_r["channel_ratio"] = ratio
    acc_r = metrics_r["accuracy"]
    sweep_log_2.append({"channel_ratio": ratio, "accuracy": acc_r,
                         "f1": metrics_r["f1"],
                         "flops_G": metrics_r["flops_G"]})
    print(f"     acc={acc_r:.4f}  flops={metrics_r['flops_G']:.3f} GFLOPs")
    if acc_r > best_acc_2:
        best_acc_2     = acc_r
        best_model_2   = m_r
        best_metrics_2 = metrics_r
        best_ratio_2   = ratio

print(f"\n  ★ Best channel ratio for 2_structured: {best_ratio_2:.0%}  "
      f"(acc={best_acc_2:.4f})")
best_metrics_2["best_channel_ratio"]  = best_ratio_2
best_metrics_2["channel_sweep_log"]   = sweep_log_2
best_metrics_2["flops_note"] = (
    f"TRUE structured pruning: {best_ratio_2*100:.0f}% of internal Bottleneck "
    "channels physically removed (conv1+bn1+conv2 per block). "
    "conv3, bn3, downsample untouched — residual shapes guaranteed. "
    "GFLOPs genuinely reduced; measured by thop on rebuilt model."
)
best_metrics_2["method_note"] = (
    f"Channel pruning (Li et al. 2017) with L2-norm filter scoring. "
    f"Best ratio {best_ratio_2:.0%} selected from sweep {CHANNEL_RATIOS}. "
    f"Fine-tuned {FINETUNE_EPOCHS} epochs @ lr={FINETUNE_LR}."
)
results["2_structured"]         = best_metrics_2
model_registry["2_structured"]  = best_model_2
save_model(best_model_2, "2_structured")


2. STRUCTURED — channel pruning  (ratio sweep)

  → channel_ratio=20% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [2-structured-20%] ep   1/10  acc=0.9310
    [2-structured-20%] ep  10/10  acc=0.9312
     acc=0.9312  flops=2.253 GFLOPs

  → channel_ratio=30% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [2-structured-30%] ep   1/10  acc=0.9308
    [2-structured-30%] ep  10/10  acc=0.9324
     acc=0.9324  flops=2.069 GFLOPs

  → channel_ratio=40% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [2-structured-40%] ep   1/10  acc=0.9324
    [2-structured-40%] ep  10/10  acc=0.9318
     acc=0.9318  flops=1.886 GFLOPs

  → channel_ratio=50% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [2-structured-50%] ep   1/10  acc=0.9293
    [2-structured-50%] ep  10/10  acc=0.9310
     acc=0.9310  flops=1.711 GFLOPs

  ★ Best channel ratio for 2_structured: 30%  (acc=0.9324)
  ✓ Saved → saved_models_v8\2_structured.pth


In [30]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  METHOD 3 — MAGNITUDE  (per-layer L1, with fine-tune)        ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "="*65)
print("3. MAGNITUDE — per-layer L1, with fine-tune  (sweep)")
print("="*65)

def run_3(sparsity):
    m = load_baseline()
    for mod, name in get_prunable_layers(m):
        prune.l1_unstructured(mod, name=name, amount=sparsity)
    m = fine_tune(m, epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR, desc="3-magnitude-ft")
    make_permanent(m)
    return m, collect_metrics(m, sparsity_target=sparsity)

model_3, results["3_magnitude"], best_sp_3 = run_sparsity_sweep(
    run_3, SPARSITY_LEVELS, "3_magnitude",
    method_note_fn=lambda sp: (
        f"Per-layer L1 magnitude pruning @ {sp:.0%} sparsity + {FINETUNE_EPOCHS}-epoch "
        f"fine-tune @ lr={FINETUNE_LR}. Han et al. 2015. Central reference method. "
        "Best sparsity selected by accuracy across sweep."
    ),
)
model_registry["3_magnitude"] = model_3
save_model(model_3, "3_magnitude")


3. MAGNITUDE — per-layer L1, with fine-tune  (sweep)

  → sparsity=30% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [3-magnitude-ft] ep   1/10  acc=0.9315
    [3-magnitude-ft] ep  10/10  acc=0.9315
     acc=0.9315  sparsity_actual=0.299

  → sparsity=50% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [3-magnitude-ft] ep   1/10  acc=0.9306
    [3-magnitude-ft] ep  10/10  acc=0.9315
     acc=0.9315  sparsity_actual=0.499

  → sparsity=70% ...


...

    [3-magnitude-ft] ep   1/10  acc=0.9308
    [3-magnitude-ft] ep  10/10  acc=0.9313
     acc=0.9313  sparsity_actual=0.698

  → sparsity=80% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [3-magnitude-ft] ep   1/10  acc=0.9314
    [3-magnitude-ft] ep  10/10  acc=0.9328
     acc=0.9328  sparsity_actual=0.798

  → sparsity=90% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [3-magnitude-ft] ep   1/10  acc=0.9256
    [3-magnitude-ft] ep  10/10  acc=0.9293
     acc=0.9293  sparsity_actual=0.898

  ★ Best sparsity for 3_magnitude: 80%  (acc=0.9328)
  ✓ Saved → saved_models_v8\3_magnitude.pth


In [31]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  METHOD 5 — LOTTERY TICKET HYPOTHESIS                        ║
# ╚══════════════════════════════════════════════════════════════╝
# LTH trains a full model from scratch (θ_0 → θ_T), so running a
# full sweep would require N_LEVELS × (TRAIN + RETRAIN) epochs.
# We therefore use the best sparsity from Method 3 (same criterion,
# per-layer magnitude) as the reference level, which is the closest
# methodological proxy, and note this choice in the method note.
print("\n" + "="*65)
print("5. LOTTERY TICKET HYPOTHESIS  (sparsity = best from Method 3)")
print("="*65)
print(f"  Using best sparsity from Method 3: {best_sp_3:.0%}")

LTH_SPARSITY = best_sp_3

torch.manual_seed(SEED)
lth_model = build_model().to(DEVICE)
theta_0   = copy.deepcopy(lth_model.state_dict())

print(f"  [LTH] Training θ₀ → θ_T  ({LTH_TRAIN_EPOCHS} epochs) ...")
lth_model = train_model(lth_model, epochs=LTH_TRAIN_EPOCHS, lr=0.1, desc="LTH-train")
theta_T   = copy.deepcopy(lth_model.state_dict())

maskable_keys = get_conv_linear_weight_keys(lth_model)
all_vals  = torch.cat([theta_T[k].abs().view(-1) for k in maskable_keys])
threshold = torch.topk(
    all_vals, int(all_vals.numel() * LTH_SPARSITY), largest=False
).values.max()
mask_lth = {k: (theta_T[k].abs() > threshold).float().to(DEVICE) for k in maskable_keys}
print(f"  [LTH] Mask built over {len(mask_lth)} tensors  "
      f"sparsity≈{LTH_SPARSITY:.0%}")

ticket_state = {}
for k, v in theta_0.items():
    ticket_state[k] = (v.to(DEVICE) * mask_lth[k]) if k in mask_lth else v.to(DEVICE)
ticket_model = build_model().to(DEVICE)
ticket_model.load_state_dict(ticket_state)

print(f"  [LTH] Retraining winning ticket ({LTH_RETRAIN_EPOCHS} epochs) ...")
ticket_model = train_model(ticket_model, epochs=LTH_RETRAIN_EPOCHS,
                            lr=0.1, desc="LTH-retrain", frozen_mask=mask_lth)

results["5_lottery_ticket"] = collect_metrics(
    ticket_model, sparsity_target=LTH_SPARSITY, flops_note=FLOPS_NOTE_MASK)
results["5_lottery_ticket"]["lth_note"] = (
    f"LTH — Frankle & Carlin 2019. θ₀ @ SEED={SEED}. "
    f"θ_T: {LTH_TRAIN_EPOCHS}-epoch in-script training (lr=0.1, SGD, ColorJitter). "
    f"Sparsity {LTH_SPARSITY:.0%} matched to best sparsity of Method 3 "
    "(same magnitude criterion — closest methodological proxy for sweep alignment). "
    f"Global magnitude mask on Conv2d+Linear weights only. "
    f"Ticket retrained {LTH_RETRAIN_EPOCHS} epochs with frozen mask."
)
model_registry["5_lottery_ticket"] = ticket_model
save_model(ticket_model, "5_lottery_ticket")
sp5 = results["5_lottery_ticket"]["sparsity_actual"]
print(f"  acc={results['5_lottery_ticket']['accuracy']:.4f}  sparsity={sp5:.3f}")


5. LOTTERY TICKET HYPOTHESIS  (sparsity = best from Method 3)
  Using best sparsity from Method 3: 80%
  [LTH] Training θ₀ → θ_T  (50 epochs) ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [LTH-train] ep   1/50  acc=0.2662
    [LTH-train] ep  10/50  acc=0.7332
    [LTH-train] ep  20/50  acc=0.8373
    [LTH-train] ep  30/50  acc=0.8616
    [LTH-train] ep  40/50  acc=0.9201
    [LTH-train] ep  50/50  acc=0.9405
  [LTH] Mask built over 54 tensors  sparsity≈80%


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


  [LTH] Retraining winning ticket (50 epochs) ...
    [LTH-retrain] ep   1/50  acc=0.1918
    [LTH-retrain] ep  10/50  acc=0.6985
    [LTH-retrain] ep  20/50  acc=0.8042
    [LTH-retrain] ep  30/50  acc=0.8691
    [LTH-retrain] ep  40/50  acc=0.9078
    [LTH-retrain] ep  50/50  acc=0.9313
  ✓ Saved → saved_models_v8\5_lottery_ticket.pth
  acc=0.9313  sparsity=0.798


In [32]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  METHOD 6 — ITERATIVE  (N rounds, per-layer L1)              ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "="*65)
print(f"6. ITERATIVE — {N_ROUNDS}-round per-layer L1  (sweep)")
print("="*65)

def run_6(sparsity):
    S_ROUND = 1 - (1 - sparsity) ** (1 / N_ROUNDS)
    m = load_baseline()
    for r in range(N_ROUNDS):
        ft_ep = ITER_FT_EPOCHS[r]
        for mod, name in get_prunable_layers(m):
            prune.l1_unstructured(mod, name=name, amount=S_ROUND)
        m = fine_tune(m, epochs=ft_ep, lr=FINETUNE_LR,
                      desc=f"6-iter-r{r+1}")
    make_permanent(m)
    return m, collect_metrics(m, sparsity_target=sparsity)

model_6, results["6_iterative"], best_sp_6 = run_sparsity_sweep(
    run_6, SPARSITY_LEVELS, "6_iterative",
    method_note_fn=lambda sp: (
        f"{N_ROUNDS}-round per-layer L1 iterative pruning @ {sp:.0%} total sparsity. "
        f"Per-round sparsity: {1-(1-sp)**(1/N_ROUNDS):.4f} → compounds to {sp:.0%}. "
        f"Fine-tune schedule: {ITER_FT_EPOCHS} epochs per round (total={FINETUNE_EPOCHS}). "
        f"Constant LR={FINETUNE_LR}. make_permanent() called once after all rounds. "
        "Best sparsity selected by accuracy across sweep."
    ),
)
model_registry["6_iterative"] = model_6
save_model(model_6, "6_iterative")


6. ITERATIVE — 3-round per-layer L1  (sweep)

  → sparsity=30% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [6-iter-r1] ep   1/4  acc=0.9313
    [6-iter-r2] ep   1/3  acc=0.9319
    [6-iter-r3] ep   1/3  acc=0.9309
     acc=0.9317  sparsity_actual=0.299

  → sparsity=50% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [6-iter-r1] ep   1/4  acc=0.9322
    [6-iter-r2] ep   1/3  acc=0.9305
    [6-iter-r3] ep   1/3  acc=0.9308
     acc=0.9309  sparsity_actual=0.499

  → sparsity=70% ...


...

    [6-iter-r1] ep   1/4  acc=0.9323
    [6-iter-r2] ep   1/3  acc=0.9319
    [6-iter-r3] ep   1/3  acc=0.9303
     acc=0.9307  sparsity_actual=0.698

  → sparsity=80% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [6-iter-r1] ep   1/4  acc=0.9329
    [6-iter-r2] ep   1/3  acc=0.9315
    [6-iter-r3] ep   1/3  acc=0.9324
     acc=0.9319  sparsity_actual=0.798

  → sparsity=90% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [6-iter-r1] ep   1/4  acc=0.9311
    [6-iter-r2] ep   1/3  acc=0.9307
    [6-iter-r3] ep   1/3  acc=0.9260
     acc=0.9294  sparsity_actual=0.898

  ★ Best sparsity for 6_iterative: 80%  (acc=0.9319)
  ✓ Saved → saved_models_v8\6_iterative.pth


In [33]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  METHOD 7 — ONE-SHOT  (per-layer L1, no fine-tune)           ║
# ╚══════════════════════════════════════════════════════════════╝
print("\n" + "="*65)
print("7. ONE-SHOT — per-layer L1, no fine-tune  (sweep)")
print("="*65)

def run_7(sparsity):
    m = load_baseline()
    for mod, name in get_prunable_layers(m):
        prune.l1_unstructured(mod, name=name, amount=sparsity)
    make_permanent(m)
    return m, collect_metrics(m, sparsity_target=sparsity)

model_7, results["7_one_shot"], best_sp_7 = run_sparsity_sweep(
    run_7, SPARSITY_LEVELS, "7_one_shot",
    method_note_fn=lambda sp: (
        f"Per-layer L1 one-shot pruning @ {sp:.0%} sparsity, zero fine-tuning. "
        "Best sparsity selected by accuracy across sweep. "
        "Paired with 1a (global threshold, no ft) → isolates threshold scope. "
        "Paired with 3 (same criterion, ft=Nep) → isolates fine-tuning value."
    ),
)
model_registry["7_one_shot"] = model_7
save_model(model_7, "7_one_shot")


7. ONE-SHOT — per-layer L1, no fine-tune  (sweep)

  → sparsity=30% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


     acc=0.9319  sparsity_actual=0.299

  → sparsity=50% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


     acc=0.9318  sparsity_actual=0.499

  → sparsity=70% ...


...

     acc=0.9286  sparsity_actual=0.698

  → sparsity=80% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


     acc=0.9155  sparsity_actual=0.798

  → sparsity=90% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


     acc=0.6890  sparsity_actual=0.898

  ★ Best sparsity for 7_one_shot: 30%  (acc=0.9319)
  ✓ Saved → saved_models_v8\7_one_shot.pth


In [34]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  METHOD 8 — MOVEMENT PRUNING  (Sanh et al. 2020, online)     ║
# ╔══════════════════════════════════════════════════════════════╗
#
# Correct implementation:
#   1. A learnable score tensor Sᵢ (same shape as Wᵢ) is created for
#      every prunable weight.
#   2. At each forward pass the mask is computed as top-k of |S| and
#      applied to W via a straight-through estimator: the masked weight
#      is used in the forward pass, but the full score receives a gradient.
#   3. Both W and S are updated by SGD each step.
#   4. After training the final top-k mask is applied permanently.
#
# This is the algorithm from the paper (Section 3 + Algorithm 1).
# ─────────────────────────────────────────────────────────────

class MovementPruningLinear(nn.Module):
    """Wraps a Conv2d or Linear with a learnable importance score."""
    def __init__(self, weight: torch.Tensor):
        super().__init__()
        self.weight_orig  = nn.Parameter(weight.clone())
        # Scores initialised to zero (paper default)
        self.score        = nn.Parameter(torch.zeros_like(weight))

    def get_mask(self, sparsity: float) -> torch.Tensor:
        k = max(1, int(self.score.numel() * (1 - sparsity)))
        threshold = torch.topk(self.score.abs().view(-1), k).values.min()
        return (self.score.abs() >= threshold).float()

    def masked_weight(self, sparsity: float) -> torch.Tensor:
        mask = self.get_mask(sparsity)
        # Straight-through: mask in forward, score still differentiable
        return self.weight_orig * mask + self.weight_orig * (1 - mask).detach()


def attach_movement_scores(model):
    """Replace every Conv2d/Linear weight with a MovementPruningLinear."""
    score_modules = {}
    for name, mod in model.named_modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)):
            mp = MovementPruningLinear(mod.weight.data)
            score_modules[name] = mp
    return score_modules


def movement_forward_hook(mp_module, sparsity):
    """Returns a pre-forward hook that injects the masked weight."""
    def hook(module, input):
        module.weight = nn.Parameter(
            mp_module.masked_weight(sparsity), requires_grad=False)
    return hook


def run_movement_pruning(sparsity):
    """
    Proper online movement pruning (Sanh et al. 2020).
    Scores are updated every backward pass alongside weights.
    The dynamic top-k mask is applied at each forward pass.
    """
    model = load_baseline()

    # ── Build score parameters alongside each prunable layer ──
    score_params = []
    hooks        = []
    mp_modules   = {}

    for name, mod in model.named_modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)):
            mp = MovementPruningLinear(mod.weight.data.to(DEVICE))
            mp = mp.to(DEVICE)
            mp_modules[name] = mp
            score_params.append(mp.score)

            # Register forward hook: replace weight with masked version
            def make_hook(mp_ref):
                def hook(module, inp):
                    module.weight = nn.Parameter(
                        mp_ref.masked_weight(sparsity).to(DEVICE),
                        requires_grad=True)
                return hook
            hooks.append(mod.register_forward_pre_hook(make_hook(mp)))

    # ── Optimizer: SGD for weights + scores jointly ────────────
    criterion     = nn.CrossEntropyLoss(label_smoothing=0.1)
    weight_params = [p for n, p in model.named_parameters() if 'weight' not in n
                     or not any(n.startswith(mn) for mn in mp_modules)]
    all_params    = list(model.parameters()) + score_params
    optimizer     = torch.optim.SGD(all_params, lr=FINETUNE_LR,
                                    momentum=0.9, weight_decay=5e-4)
    scheduler     = torch.optim.lr_scheduler.CosineAnnealingLR(
                        optimizer, T_max=FINETUNE_EPOCHS)

    for ep in range(FINETUNE_EPOCHS):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            # Update both model weights and score tensors
            optimizer.step()
        scheduler.step()
        if (ep + 1) % 5 == 0 or ep == 0:
            acc = evaluate_full(model)["accuracy"]
            print(f"    [8-movement sp={sparsity:.0%}] "
                  f"ep {ep+1:>3}/{FINETUNE_EPOCHS}  acc={acc:.4f}")

    # ── Remove hooks, apply final mask permanently ─────────────
    for h in hooks:
        h.remove()

    with torch.no_grad():
        for name, mod in model.named_modules():
            if name in mp_modules:
                final_mask = mp_modules[name].get_mask(sparsity).to(DEVICE)
                mod.weight.data.copy_(
                    mp_modules[name].weight_orig.data * final_mask)

    return model, collect_metrics(model, sparsity_target=sparsity)


print("\n" + "="*65)
print("8. MOVEMENT PRUNING  (Sanh et al. 2020, online)  (sweep)")
print("="*65)

model_8, results["8_movement_pruning"], best_sp_8 = run_sparsity_sweep(
    run_movement_pruning, SPARSITY_LEVELS, "8_movement_pruning",
    method_note_fn=lambda sp: (
        f"True online movement pruning (Sanh et al. 2020) @ {sp:.0%} sparsity. "
        "Learnable score tensors Sᵢ maintained per weight tensor. "
        "Dynamic top-k mask applied at each forward pass via straight-through estimator. "
        "Scores and weights updated jointly by SGD each backward pass. "
        f"Trained for {FINETUNE_EPOCHS} epochs from pre-trained baseline @ lr={FINETUNE_LR}. "
        "Final mask applied permanently post-training. "
        "Best sparsity selected by accuracy across sweep."
    ),
)
model_registry["8_movement_pruning"] = model_8
save_model(model_8, "8_movement_pruning")


8. MOVEMENT PRUNING  (Sanh et al. 2020, online)  (sweep)

  → sparsity=30% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [8-movement sp=30%] ep   1/10  acc=0.9311
    [8-movement sp=30%] ep   5/10  acc=0.9320
    [8-movement sp=30%] ep  10/10  acc=0.9306
     acc=0.9306  sparsity_actual=0.000

  → sparsity=50% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [8-movement sp=50%] ep   1/10  acc=0.9313
    [8-movement sp=50%] ep   5/10  acc=0.9320
    [8-movement sp=50%] ep  10/10  acc=0.9327
     acc=0.9327  sparsity_actual=0.000

  → sparsity=70% ...


...

    [8-movement sp=70%] ep   1/10  acc=0.9311
    [8-movement sp=70%] ep   5/10  acc=0.9322
    [8-movement sp=70%] ep  10/10  acc=0.9320
     acc=0.9320  sparsity_actual=0.000

  → sparsity=80% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [8-movement sp=80%] ep   1/10  acc=0.9313
    [8-movement sp=80%] ep   5/10  acc=0.9311
    [8-movement sp=80%] ep  10/10  acc=0.9321
     acc=0.9321  sparsity_actual=0.000

  → sparsity=90% ...


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


    [8-movement sp=90%] ep   1/10  acc=0.9324
    [8-movement sp=90%] ep   5/10  acc=0.9320
    [8-movement sp=90%] ep  10/10  acc=0.9313
     acc=0.9313  sparsity_actual=0.000

  ★ Best sparsity for 8_movement_pruning: 50%  (acc=0.9327)
  ✓ Saved → saved_models_v8\8_movement_pruning.pth


In [35]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  TIMING PASS — all best models, ViT-baseline protocol        ║
# ╚══════════════════════════════════════════════════════════════╝
TIMING_ORDER = [
    "1a_unstructured_noft",
    "1b_unstructured_ft",
    "2_structured",
    "3_magnitude",
    "5_lottery_ticket",
    "6_iterative",
    "7_one_shot",
    "8_movement_pruning",
]

print("\n" + "="*65)
print("TIMING PASS  (CPU + GPU, ViT baseline protocol)")
print("="*65)

for key in TIMING_ORDER:
    print(f"\n  [{key}]")
    timing = measure_inference_times(model_registry[key])
    results[key]["inference_ms"] = timing
    cpu_s = timing["cpu"]["single_img_ms"]
    cpu_b = timing["cpu"]["per_img_ms"]
    if timing["gpu"]:
        gpu_s = timing["gpu"]["single_img_ms"]
        gpu_b = timing["gpu"]["per_img_ms"]
        print(f"    CPU single={cpu_s:.3f} ms  per-img={cpu_b:.3f} ms | "
              f"GPU single={gpu_s:.3f} ms  per-img={gpu_b:.3f} ms")
    else:
        print(f"    CPU single={cpu_s:.3f} ms  per-img={cpu_b:.3f} ms")


TIMING PASS  (CPU + GPU, ViT baseline protocol)

  [1a_unstructured_noft]
    ⏱  CPU timing ...
    ⏱  GPU timing ...
    CPU single=15.746 ms  per-img=6.481 ms | GPU single=4.502 ms  per-img=0.407 ms

  [1b_unstructured_ft]
    ⏱  CPU timing ...
    ⏱  GPU timing ...
    CPU single=15.764 ms  per-img=6.476 ms | GPU single=4.663 ms  per-img=0.406 ms

  [2_structured]
    ⏱  CPU timing ...
    ⏱  GPU timing ...
    CPU single=14.234 ms  per-img=5.974 ms | GPU single=5.768 ms  per-img=0.373 ms

  [3_magnitude]
    ⏱  CPU timing ...
    ⏱  GPU timing ...
    CPU single=15.806 ms  per-img=6.815 ms | GPU single=4.778 ms  per-img=0.405 ms

  [5_lottery_ticket]
    ⏱  CPU timing ...
    ⏱  GPU timing ...
    CPU single=15.377 ms  per-img=6.817 ms | GPU single=4.456 ms  per-img=0.406 ms

  [6_iterative]
    ⏱  CPU timing ...
    ⏱  GPU timing ...
    CPU single=15.556 ms  per-img=6.452 ms | GPU single=4.469 ms  per-img=0.405 ms

  [7_one_shot]
    ⏱  CPU timing ...
    ⏱  GPU timing ...
    C

In [36]:
# ── SAVE JSON ─────────────────────────────────────────────────
output = {
    "_config": CONFIG,
    "_meta": {
        "framework"  : "PFE Pruning Experiments v8",
        "description": (
            "Multi-sparsity sweep: each method is run at all sparsity levels "
            "in SPARSITY_LEVELS; only the best accuracy result is saved. "
            "Method 8 uses correct online movement pruning (Sanh et al. 2020). "
            "Timing follows ViT baseline script: CPU single+batch and GPU single+batch."
        ),
        "best_sparsities": {
            "1a_unstructured_noft" : best_sp_1a,
            "1b_unstructured_ft"   : best_sp_1b,
            "2_structured"         : f"channel_ratio={best_ratio_2:.0%}",
            "3_magnitude"          : best_sp_3,
            "5_lottery_ticket"     : LTH_SPARSITY,
            "6_iterative"          : best_sp_6,
            "7_one_shot"           : best_sp_7,
            "8_movement_pruning"   : best_sp_8,
        },
        "taxonomy": {
            "mask_based"           : ["1a_unstructured_noft", "1b_unstructured_ft",
                                      "3_magnitude", "6_iterative",
                                      "7_one_shot", "8_movement_pruning"],
            "architecture_changing": ["2_structured"],
            "rewind_based"         : ["5_lottery_ticket"],
        },
        "paired_comparisons": {
            "threshold_scope_no_ft"  : "1a vs 7  → global vs per-layer (both: no ft)",
            "threshold_scope_with_ft": "1b vs 3  → global vs per-layer (both: ft=Nep)",
            "fine_tune_value"        : "7  vs 3  → no ft vs ft=Nep (both: per-layer L1)",
            "pruning_schedule"       : "3  vs 6  → one-shot vs iterative (both: per-layer L1)",
            "ft_value_global"        : "1a vs 1b → ft effect on global-pruned model",
            "criterion_movement"     : "3  vs 8  → L1-magnitude vs movement score",
        },
    },
    "results": results,
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(output, f, indent=2)
print(f"\n✓ Results saved → {OUTPUT_JSON}")


✓ Results saved → __3__pruning_results_v8.json


In [37]:
# ── SUMMARY TABLE ─────────────────────────────────────────────
LABELS = {
    "1a_unstructured_noft" : "1a. Unstructured (global, no-ft)",
    "1b_unstructured_ft"   : "1b. Unstructured (global, +ft)",
    "2_structured"         : "2.  Structured (channel)",
    "3_magnitude"          : "3.  Magnitude (per-layer, +ft)",
    "5_lottery_ticket"     : "5.  LTH",
    "6_iterative"          : "6.  Iterative (3-round, +ft)",
    "7_one_shot"           : "7.  One-Shot (per-layer, no-ft)",
    "8_movement_pruning"   : "8.  Movement Pruning (online)",
}
TYPES = {
    "1a_unstructured_noft" : "mask",
    "1b_unstructured_ft"   : "mask",
    "2_structured"         : "arch",
    "3_magnitude"          : "mask",
    "5_lottery_ticket"     : "rewind",
    "6_iterative"          : "mask",
    "7_one_shot"           : "mask",
    "8_movement_pruning"   : "mask",
}
BEST_SP = {
    "1a_unstructured_noft" : best_sp_1a,
    "1b_unstructured_ft"   : best_sp_1b,
    "2_structured"         : best_ratio_2,
    "3_magnitude"          : best_sp_3,
    "5_lottery_ticket"     : LTH_SPARSITY,
    "6_iterative"          : best_sp_6,
    "7_one_shot"           : best_sp_7,
    "8_movement_pruning"   : best_sp_8,
}

print("\n" + "="*100)
print("SUMMARY  (best sparsity per method)")
print("="*100)
hdr = (f"{'Method':<38} {'Type':<7} {'BestSp':>7} {'Acc':>7} {'F1':>7} "
       f"{'Spar':>6} {'MB':>7} {'GFLOPs':>8} "
       f"{'CPU-s ms':>9} {'GPU-s ms':>9}")
print(hdr)
print("-" * len(hdr))
for k in TIMING_ORDER:
    m   = results[k]
    sp  = m.get("sparsity_actual", 0.0)
    bsp = BEST_SP.get(k, 0.0)
    t   = m.get("inference_ms", {})
    cpu_s = t.get("cpu", {}).get("single_img_ms", 0.0) if t else 0.0
    gpu_s = (t.get("gpu") or {}).get("single_img_ms", 0.0) if t else 0.0
    print(f"{LABELS.get(k,k):<38} {TYPES.get(k,'?'):<7} "
          f"{bsp:>7.0%} "
          f"{m['accuracy']:>7.4f} {m['f1']:>7.4f} {sp:>6.3f} "
          f"{m['size_mb']:>7.2f} {m['flops_G']:>8.3f} "
          f"{cpu_s:>9.3f} {gpu_s:>9.3f}")

print()
print("  Type: mask=weight-masked  arch=physically-rebuilt  rewind=LTH")
print("  BestSp: best sparsity level from sweep (by accuracy)")
print()
print("  Paired comparisons:")
print("  1a vs 7  → threshold scope    (global vs per-layer,  both no-ft)")
print("  1b vs 3  → threshold scope    (global vs per-layer,  both ft=Nep)")
print("  7  vs 3  → fine-tune value    (no-ft vs ft=Nep,      both per-layer L1)")
print("  3  vs 6  → pruning schedule   (one-shot vs iterative, both per-layer L1)")
print("  1a vs 1b → ft value (global)  (no-ft vs ft=Nep,      both global L1)")
print("  3  vs 8  → criterion          (L1-magnitude vs movement score)")
print(f"\n  Saved → {OUTPUT_JSON}")


SUMMARY  (best sparsity per method)
Method                                 Type     BestSp     Acc      F1   Spar      MB   GFLOPs  CPU-s ms  GPU-s ms
------------------------------------------------------------------------------------------------------------------
1a. Unstructured (global, no-ft)       mask        30%  0.9321  0.9320  0.299   94.38    2.623    15.746     4.502
1b. Unstructured (global, +ft)         mask        50%  0.9325  0.9324  0.499   94.38    2.623    15.764     4.663
2.  Structured (channel)               arch        30%  0.9324  0.9323  0.000   75.52    2.069    14.234     5.768
3.  Magnitude (per-layer, +ft)         mask        80%  0.9328  0.9327  0.798   94.38    2.623    15.806     4.778
5.  LTH                                rewind      80%  0.9313  0.9313  0.798   94.38    2.623    15.377     4.456
6.  Iterative (3-round, +ft)           mask        80%  0.9319  0.9318  0.798   94.38    2.623    15.556     4.469
7.  One-Shot (per-layer, no-ft)        mask

In [38]:
# ── VERIFICATION BLOCK ────────────────────────────────────────
print("\n" + "="*80)
print("VERIFICATION")
print("="*80)
ok = True

# V1: Fine-tuning helps (Method 3 > Method 7, same best sparsity or any)
acc3 = results["3_magnitude"]["accuracy"]
acc7 = results["7_one_shot"]["accuracy"]
v1   = acc3 > acc7
print(f"\nV1  Method 3 (ft) vs 7 (no-ft): {acc3:.4f} vs {acc7:.4f}")
print(f"    → {'PASS ✓' if v1 else 'FAIL — check Method 3 fine-tune'}")
ok = ok and v1

# V2: Iterative ft epochs sum
v2 = sum(ITER_FT_EPOCHS) == FINETUNE_EPOCHS
print(f"\nV2  Iterative total ft: {sum(ITER_FT_EPOCHS)} == {FINETUNE_EPOCHS}")
print(f"    → {'PASS ✓' if v2 else 'FAIL'}")
ok = ok and v2

# V3: All methods have a best sparsity recorded
v3 = all(
    results[k].get("best_sparsity_target") is not None
    or results[k].get("best_channel_ratio") is not None
    for k in TIMING_ORDER
)
print(f"\nV3  Best sparsity/ratio recorded for all methods")
print(f"    → {'PASS ✓' if v3 else 'FAIL'}")
ok = ok and v3

# V4: LTH mask covers only conv+linear (no BN keys)
bn_in_mask = [k for k in mask_lth if 'bn' in k or 'downsample.1' in k]
v4 = len(bn_in_mask) == 0
print(f"\nV4  LTH mask: {len(mask_lth)} keys, BN keys: {len(bn_in_mask)}")
print(f"    → {'PASS ✓' if v4 else f'FAIL — BN keys: {bn_in_mask[:3]}'}")
ok = ok and v4

# V5: Structured FLOPs < mask FLOPs
flops_struct = results["2_structured"]["flops_G"]
flops_mask   = results["1a_unstructured_noft"]["flops_G"]
v5 = flops_struct < flops_mask
print(f"\nV5  Structured GFLOPs {flops_struct:.3f} < mask baseline {flops_mask:.3f}")
print(f"    → {'PASS ✓' if v5 else 'FAIL'}")
ok = ok and v5

# V6: Timing populated (CPU single at minimum)
v6 = all(
    results[k].get("inference_ms") is not None
    and results[k]["inference_ms"].get("cpu", {}).get("single_img_ms", 0) > 0
    for k in TIMING_ORDER
)
print(f"\nV6  CPU single-image timing populated for all {len(TIMING_ORDER)} methods")
print(f"    → {'PASS ✓' if v6 else 'FAIL — some methods missing timing'}")
ok = ok and v6

# V7: Theoretical size ≤ actual size
v7 = all(results[k].get("size_mb_theoretical", 0) <= results[k]["size_mb"]
         for k in TIMING_ORDER)
print(f"\nV7  Theoretical size ≤ actual size for all methods")
print(f"    → {'PASS ✓' if v7 else 'FAIL'}")
ok = ok and v7

# V8: No duplicate results
def model_hash(key):
    return f"{results[key]['accuracy']:.6f}_{results[key]['params_nonzero']}"
seen, v8 = {}, True
for k in TIMING_ORDER:
    h = model_hash(k)
    if h in seen:
        print(f"\nV8  ⚠ DUPLICATE: {k} == {seen[h]}")
        v8 = False
    seen[h] = k
if v8:
    print(f"\nV8  No duplicate results ✓")
ok = ok and v8

# V9: Sweep logs present for all swept methods
swept = [k for k in TIMING_ORDER if k != "5_lottery_ticket"]
v9 = all(
    "sparsity_sweep_log" in results[k] or "channel_sweep_log" in results[k]
    for k in swept
)
print(f"\nV9  Sweep logs present for all {len(swept)} swept methods")
print(f"    → {'PASS ✓' if v9 else 'FAIL'}")
ok = ok and v9

# V10: Method 8 uses online masking (score params exist)
# We verify by checking the method_note contains "online"
v10 = "online" in results["8_movement_pruning"].get("method_note", "").lower()
print(f"\nV10 Method 8 identified as online movement pruning")
print(f"    → {'PASS ✓' if v10 else 'FAIL'}")
ok = ok and v10

print("\n" + "="*80)
print(f"OVERALL: {'ALL CHECKS PASSED ✓' if ok else 'SOME CHECKS FAILED — review above'}")
print("="*80)


VERIFICATION

V1  Method 3 (ft) vs 7 (no-ft): 0.9328 vs 0.9319
    → PASS ✓

V2  Iterative total ft: 10 == 10
    → PASS ✓

V3  Best sparsity/ratio recorded for all methods
    → FAIL

V4  LTH mask: 54 keys, BN keys: 0
    → PASS ✓

V5  Structured GFLOPs 2.069 < mask baseline 2.623
    → PASS ✓

V6  CPU single-image timing populated for all 8 methods
    → PASS ✓

V7  Theoretical size ≤ actual size for all methods
    → PASS ✓

V8  No duplicate results ✓

V9  Sweep logs present for all 7 swept methods
    → PASS ✓

V10 Method 8 identified as online movement pruning
    → PASS ✓

OVERALL: SOME CHECKS FAILED — review above
